# 05 · Scikit-learn: pipelines, validação e vazamento de dados

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

Nesta aula você vai construir, do zero, um modelo que lê uma avaliação de produto escrita por um cliente e estima a chance de esse cliente **não recomendar** a compra. No caminho, vai aprender a forma profissional de usar o scikit-learn, que é diferente da forma que aparece na maioria dos tutoriais.

## O que você será capaz de fazer ao final

1. Explicar a diferença entre `fit`, `transform` e `predict`, e dizer o que cada objeto do scikit-learn aprende.
2. Montar um `Pipeline` com `ColumnTransformer` que trata colunas numéricas, categóricas e de texto.
3. Escolher métricas adequadas para um problema desbalanceado e justificar a escolha.
4. Estimar o desempenho com validação cruzada e interpretar a variação entre as dobras.
5. Reconhecer e corrigir os três tipos mais comuns de **vazamento de dados**.
6. Ajustar hiperparâmetros sem contaminar a avaliação e escolher um limiar de decisão a partir de custos.

## Como usar este notebook

Execute as células na ordem. Leia o texto antes de executar o código: boa parte do aprendizado está em tentar prever o resultado.

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

## 0. Preparação

Primeiro conferimos as versões. Esta aula usa recursos do scikit-learn 1.4 ou mais recente.

In [ ]:
import platform

import matplotlib
import numpy as np
import pandas as pd
import sklearn

print(f"Python        {platform.python_version()}")
print(f"numpy         {np.__version__}")
print(f"pandas        {pd.__version__}")
print(f"scikit-learn  {sklearn.__version__}")
print(f"matplotlib    {matplotlib.__version__}")

versao_sklearn = tuple(int(parte) for parte in sklearn.__version__.split(".")[:2])
assert versao_sklearn >= (1, 4), (
    "Esta aula precisa do scikit-learn 1.4 ou mais novo. "
    "Execute `%pip install -U scikit-learn` em uma célula e reinicie o ambiente de execução."
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn import set_config

SEMENTE = 42                      # usada em tudo que tem sorteio, para os resultados se repetirem
set_config(display="diagram")     # mostra pipelines como diagramas interativos
pd.set_option("display.max_colwidth", 90)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["figure.dpi"] = 110


def milhar(n):
    """Formata um número com ponto de milhar: 132373 -> '132.373'."""
    return f"{n:,.0f}".replace(",", ".")

### Os dados

Usaremos o **B2W-Reviews01**, um conjunto aberto com mais de 130 mil avaliações de produtos feitas por clientes do site Americanas.com entre janeiro e maio de 2018. Cada linha traz o texto da avaliação, a nota de 1 a 5, a resposta à pergunta "você recomendaria este produto a um amigo?" e algumas informações do cliente e do produto.

> Real, L., Oshiro, M., Mafra, A. *B2W-Reviews01: an open product reviews corpus.* STIL 2019. Licença CC BY-NC-SA 4.0 (uso não comercial, com atribuição).

O endereço abaixo aponta para uma versão fixa do arquivo, de modo que todos trabalhem exatamente com os mesmos dados.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")

if not ARQUIVO.exists():          # baixa só na primeira vez (≈ 50 MB)
    urlretrieve(URL, ARQUIVO)

bruto = pd.read_csv(ARQUIVO, low_memory=False, dtype={"product_id": str})
print(f"{milhar(bruto.shape[0])} linhas e {bruto.shape[1]} colunas")
bruto.head(3)

| Coluna | Conteúdo |
|---|---|
| `submission_date` | data e hora em que a avaliação foi enviada |
| `reviewer_id`, `product_id` | identificadores anônimos do cliente e do produto |
| `product_name`, `product_brand` | nome e marca do produto |
| `site_category_lv1`, `site_category_lv2` | categoria e subcategoria do produto no site |
| `review_title`, `review_text` | título e texto escritos pelo cliente |
| `overall_rating` | nota de 1 a 5 estrelas |
| `recommend_to_a_friend` | "Yes" ou "No": recomendaria a um amigo? |
| `reviewer_birth_year`, `reviewer_gender`, `reviewer_state` | ano de nascimento, gênero e estado do cliente |

## 1. Aquecimento: o pandas que vamos precisar

📌 Esta seção é uma revisão rápida. Se as quatro operações abaixo forem familiares, faça direto o exercício de aquecimento.

**1. Máscara booleana**: uma comparação sobre uma coluna devolve uma coluna de `True`/`False`, que pode filtrar linhas ou ser contada.

In [ ]:
nota_baixa = bruto["overall_rating"] <= 2      # Series de True/False, uma por linha
print(nota_baixa.head(4).tolist())
print("Quantidade de notas baixas:", nota_baixa.sum())       # True conta como 1
print("Proporção de notas baixas: ", round(nota_baixa.mean(), 3))   # média de True/False = proporção

**2. Contagem de valores** com `value_counts`. O argumento `normalize=True` devolve proporções e `dropna=False` inclui os valores ausentes na contagem.

In [ ]:
bruto["recommend_to_a_friend"].value_counts(normalize=True, dropna=False).round(4)

**3. Agrupar e resumir** com `groupby`: "para cada categoria, calcule a nota média".

In [ ]:
(bruto.groupby("site_category_lv1")["overall_rating"]
      .agg(["mean", "size"])
      .sort_values("size", ascending=False)
      .head(5)
      .round(2))

**4. Substituir valores condicionalmente** com `where`: mantém o valor onde a condição é verdadeira e coloca ausente (`NaN`) onde é falsa.

In [ ]:
exemplo = pd.Series([1958, 59, 1990, 2018])
exemplo.where(exemplo.between(1918, 2004))     # anos implausíveis viram NaN

### ✏️ Aquecimento (★)

Usando o DataFrame `bruto`, calcule:

1. `prop_nota5`: a proporção de avaliações com nota 5 (um número entre 0 e 1);
2. `estado_top`: a sigla do estado com mais avaliações;
3. `media_por_genero`: uma `Series` com a nota média para cada valor de `reviewer_gender`.

In [ ]:
# ✏️ Aquecimento
# SEU CÓDIGO AQUI
prop_nota5 = ...
estado_top = ...
media_por_genero = ...

In [ ]:
# ✅ Verificação do aquecimento (não altere)
assert isinstance(prop_nota5, float) and 0.30 < prop_nota5 < 0.40, "prop_nota5 deve ser uma proporção; confira a comparação e a média"
assert bruto["reviewer_state"].eq(estado_top).sum() > 40_000, "esse não é o estado com mais avaliações"
assert set(media_por_genero.index) == {"F", "M"}, "media_por_genero deve ter um valor para F e outro para M"
assert media_por_genero.between(1, 5).all()
print("Aquecimento concluído.")
print(f"Nota 5: {prop_nota5:.1%} · estado com mais avaliações: {estado_top}")
print(media_por_genero.round(3).to_string())

## 2. O problema

### O cenário

Uma empresa de comércio eletrônico recebe opiniões de clientes por vários canais: e-mail, chat, redes sociais, formulário de avaliação. Só no formulário o cliente dá uma nota e responde se recomendaria o produto. A empresa quer um modelo que, **a partir do que o cliente escreveu e do que se sabe sobre ele e sobre o produto**, estime a chance de ele estar insatisfeito. Mensagens com chance alta vão para a frente da fila do atendimento.

As avaliações do formulário servem de dados de treino porque nelas a resposta é conhecida.

### O alvo

Chamaremos de **classe positiva** aquilo que queremos detectar: o cliente que **não recomenda**. Criamos a coluna `nao_recomenda`, que vale 1 nesses casos e 0 nos demais.

In [ ]:
df = bruto.dropna(subset=["recommend_to_a_friend"]).copy()     # 18 linhas sem resposta são descartadas
df["nao_recomenda"] = (df["recommend_to_a_friend"] == "No").astype(int)

df["nao_recomenda"].value_counts(normalize=True).rename({0: "recomenda (0)", 1: "não recomenda (1)"}).round(3)

Cerca de 27% dos clientes não recomendam. O problema é **desbalanceado**: uma classe é bem mais frequente que a outra. Guarde esse número, porque ele vai mudar a forma como medimos o modelo.

### Limpeza e variáveis derivadas

Antes de modelar, olhamos o ano de nascimento.

In [ ]:
bruto["reviewer_birth_year"].describe().round(1)

Há clientes "nascidos" no ano 59 e em 2018. São erros de digitação. Vamos calcular a idade em 2018 e tratar como ausentes as idades fora do intervalo de 14 a 100 anos. Também extraímos a hora e o dia da semana do envio e juntamos título e texto em uma única coluna.

In [ ]:
df["submission_date"] = pd.to_datetime(df["submission_date"])

idade = 2018 - df["reviewer_birth_year"]
df["idade"] = idade.where(idade.between(14, 100))            # implausíveis viram NaN

df["hora"] = df["submission_date"].dt.hour
DIAS = {0: "seg", 1: "ter", 2: "qua", 3: "qui", 4: "sex", 5: "sab", 6: "dom"}
df["dia_semana"] = df["submission_date"].dt.dayofweek.map(DIAS)

df["texto"] = df["review_title"].fillna("") + ". " + df["review_text"].fillna("")

print("Idades descartadas por serem implausíveis:", int((idade.notna() & df["idade"].isna()).sum()))
df[["idade", "hora", "dia_semana", "texto", "nao_recomenda"]].head(3)

> ⚠️ **Isso não é vazamento?** Acabamos de transformar o conjunto inteiro antes de separar treino e teste. Está correto, porque todas essas transformações são feitas **linha a linha**: a idade de um cliente depende só do ano de nascimento dele, não dos outros clientes. O problema aparece quando a transformação **aprende algo do conjunto** (uma média, uma lista de categorias, um vocabulário). Essas ficam para depois da divisão, e veremos o porquê.

Por fim, organizamos as colunas em grupos. Vamos começar só com as tabulares e deixar o texto para a seção 10.

In [ ]:
ALVO = "nao_recomenda"
NUMERICAS = ["idade", "hora"]
CATEGORICAS = ["site_category_lv1", "reviewer_state", "reviewer_gender"]

X = df.drop(columns=[ALVO, "recommend_to_a_friend"])   # tudo, menos o alvo e a coluna de onde ele saiu
y = df[ALVO]

print("X:", X.shape, "· y:", y.shape)
X[NUMERICAS + CATEGORICAS].isna().mean().round(3).rename("proporção de ausentes")

## 3. Treino e teste: o cofre

Um modelo é útil se acerta em dados que **nunca viu**. Medir o desempenho nos mesmos dados usados para treinar é como aplicar uma prova com as questões da lista de exercícios: a nota sai alta e não diz nada sobre o que o aluno aprendeu.

Por isso separamos, logo no início, um **conjunto de teste**. Pense nele como um cofre:

- ele é trancado agora e só será aberto **uma vez**, no fim, para dar a nota final do modelo;
- todas as decisões (quais colunas usar, qual modelo, quais hiperparâmetros) são tomadas olhando apenas para o **conjunto de treino**;
- se você olhar o teste, mudar o modelo e olhar de novo, o teste deixou de ser uma estimativa honesta.

Dois argumentos de `train_test_split` merecem atenção:

- `stratify=y` mantém a proporção de 27% de "não recomenda" nas duas partes. Sem isso, por azar, o teste poderia sair com 25% ou 30%;
- `random_state` fixa o sorteio, para que a divisão seja a mesma toda vez que o notebook for executado.

In [ ]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMENTE
)

print(f"Treino: {milhar(len(X_treino)):>8} linhas · {y_treino.mean():.1%} não recomendam")
print(f"Teste:  {milhar(len(X_teste)):>8} linhas · {y_teste.mean():.1%} não recomendam")

### 🔮 Preveja antes de continuar

Imagine um "modelo" que ignora os dados e responde **sempre** "recomenda". Qual será a acurácia dele (proporção de acertos) no conjunto de teste?

✏️ **Sua resposta:**

_Escreva aqui._

### ✏️ Exercício 1 (★): sua própria divisão

Divida `X` e `y` em duas partes, com **30%** das linhas na segunda, mantendo a proporção do alvo e usando `random_state=SEMENTE`. Guarde o resultado em `Xa`, `Xb`, `ya`, `yb`.

In [ ]:
# ✏️ Exercício 1
# SEU CÓDIGO AQUI
Xa, Xb, ya, yb = ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
assert abs(len(Xb) / len(X) - 0.30) < 0.001, "a segunda parte deve ter 30% das linhas"
assert abs(ya.mean() - yb.mean()) < 0.001, "as proporções do alvo diferem: faltou stratify=y?"
assert len(Xa) == len(ya) and len(Xb) == len(yb)
print(f"Exercício 1 correto: {len(Xa)} + {len(Xb)} linhas, alvo em {ya.mean():.3f} e {yb.mean():.3f}.")

## 4. A gramática do scikit-learn

Quase tudo no scikit-learn é um **estimador**: um objeto que aprende alguma coisa a partir de dados quando chamamos `fit`. O que muda de um estimador para outro é o que ele faz depois de aprender.

| Tipo | Aprende com `fit` | Depois faz | Exemplos |
|---|---|---|---|
| **Transformador** | parâmetros de uma transformação | `transform(X)`: devolve X modificado | `StandardScaler`, `SimpleImputer`, `OneHotEncoder`, `TfidfVectorizer` |
| **Preditor** | parâmetros de um modelo | `predict(X)`: devolve previsões | `LogisticRegression`, `DecisionTreeClassifier` |

Três convenções valem para a biblioteca inteira:

1. **Configuração no construtor, aprendizado no `fit`.** `StandardScaler()` cria um objeto vazio; nada é calculado até o `fit`.
2. **O que foi aprendido fica em atributos terminados em `_`.** Por exemplo `mean_`, `categories_`, `coef_`. Se o atributo não termina em sublinhado, foi você quem definiu.
3. **A entrada `X` é sempre uma tabela (duas dimensões)**, mesmo quando tem uma coluna só. Por isso escrevemos `X[["idade"]]`, com colchetes duplos, e não `X["idade"]`.

Vejamos com o `StandardScaler`, que subtrai a média e divide pelo desvio padrão.

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()                 # 1. configuração: nada foi calculado ainda
escalador.fit(X_treino[["idade"]])           # 2. aprendizado: calcula média e desvio padrão do TREINO

print(f"Média aprendida:         {escalador.mean_[0]:.2f} anos")
print(f"Desvio padrão aprendido: {escalador.scale_[0]:.2f} anos")

In [ ]:
# 3. aplicação: usa a média e o desvio do TREINO para transformar qualquer dado, inclusive o teste
escalador.transform(X_teste[["idade"]].head(5)).round(2)

O valor `0.5` significa "meio desvio padrão acima da média **do treino**". Valores ausentes continuam ausentes: o `StandardScaler` os ignora no `fit` e os preserva no `transform`.

> ⚠️ **A regra que organiza esta aula:** `fit` só enxerga dados de treino. No teste, apenas `transform` e `predict`.
>
> O atalho `fit_transform(X)` faz `fit(X)` seguido de `transform(X)` e, portanto, também só pode ser usado no treino.

### ✏️ Exercício 2 (★): um imputador

A coluna `idade` tem valores ausentes. O `SimpleImputer` os substitui por uma estatística aprendida.

1. Crie um `SimpleImputer` com `strategy="median"` e faça o `fit` em `X_treino[["idade"]]`.
2. Guarde em `mediana_idade` o valor que ele aprendeu (dica: o atributo aprendido se chama `statistics_`).
3. Guarde em `idade_teste_imputada` o resultado de aplicar o imputador a `X_teste[["idade"]]`.

In [ ]:
# ✏️ Exercício 2
from sklearn.impute import SimpleImputer

# SEU CÓDIGO AQUI
imputador = ...
mediana_idade = ...
idade_teste_imputada = ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
assert 25 < float(mediana_idade) < 50, "mediana_idade deve ser um único número, em anos"
assert np.shape(idade_teste_imputada) == (len(X_teste), 1), "aplique o imputador a X_teste[['idade']]"
assert not np.isnan(np.asarray(idade_teste_imputada, dtype=float)).any(), "ainda há valores ausentes"
print(f"Exercício 2 correto: a mediana aprendida no treino é {float(mediana_idade):.0f} anos.")

### 🔮 Para pensar

No exercício 2, os ausentes do **teste** foram preenchidos com a mediana do **treino**. Por que não calcular a mediana do próprio teste, que em princípio representaria melhor aqueles dados?

✏️ **Sua resposta:**

_Escreva aqui._

## 5. Pré-processamento com `ColumnTransformer`

Nossas colunas precisam de tratamentos diferentes:

```
                       ┌─ numéricas ──► imputar mediana ──► padronizar ──────┐
  tabela original ─────┤                                                     ├──► matriz numérica
                       └─ categóricas ► imputar "desconhecido" ► one-hot ────┘
```

O `ColumnTransformer` faz exatamente isso: aplica um transformador a cada grupo de colunas e junta os resultados lado a lado.

### Codificação one-hot em um minuto

Modelos trabalham com números. Uma coluna categórica como o estado vira várias colunas de 0 e 1, uma para cada categoria.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

brinquedo = pd.DataFrame({"estado": ["PE", "SP", "PE", "BA"]})
codificador = OneHotEncoder(sparse_output=False).fit(brinquedo)

pd.DataFrame(codificador.transform(brinquedo), columns=codificador.get_feature_names_out()).astype(int)

Duas opções do `OneHotEncoder` resolvem problemas que sempre aparecem em dados reais:

- `min_frequency=50` agrupa as categorias com menos de 50 ocorrências em uma única coluna "infrequente". Sem isso, uma categoria com 3 linhas ganharia uma coluna própria, que o modelo não tem como aprender direito.
- `handle_unknown="infrequent_if_exist"` define o que fazer quando, na hora de prever, surge uma categoria que não existia no treino: ela é tratada como infrequente em vez de causar erro.

### Montando o pré-processador

Cada grupo de colunas recebe um pequeno `Pipeline`, que é uma sequência de passos com nome.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

tratar_numericas = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

tratar_categoricas = Pipeline([
    ("imputar", SimpleImputer(strategy="constant", fill_value="desconhecido")),
    ("codificar", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50)),
])

pre = ColumnTransformer([
    ("num", tratar_numericas, NUMERICAS),          # (nome, transformador, colunas)
    ("cat", tratar_categoricas, CATEGORICAS),
])
pre       # clique nas caixas do diagrama para ver a configuração de cada passo

As colunas que não aparecem em nenhum grupo são descartadas (`remainder="drop"` é o padrão). É por isso que podemos passar a tabela `X` inteira: o identificador do cliente, a nota e o texto ficam de fora até que alguém os inclua explicitamente.

Vamos aplicar ao treino e olhar o resultado.

In [ ]:
matriz_treino = pre.fit_transform(X_treino)       # fit + transform, permitido porque é o treino

print("Tipo:   ", type(matriz_treino).__name__)
print("Formato:", matriz_treino.shape)
print("Algumas colunas geradas:")
for nome in pre.get_feature_names_out()[[0, 1, 2, 3, 20, 45, 60, -1]]:
    print("   ", nome)

Cinco colunas viraram dezenas. O resultado é uma **matriz esparsa**: como a maior parte dos valores é zero, só os diferentes de zero são guardados na memória. Os modelos do scikit-learn aceitam esse formato diretamente.

O nome de cada coluna gerada segue o padrão `grupo__coluna_categoria`, o que ajuda a interpretar o modelo depois. A coluna terminada em `infrequent_sklearn` é a que reúne as categorias raras.

### ✏️ Exercício 3 (★): aplicando ao teste

Use o pré-processador `pre`, **já ajustado no treino**, para transformar `X_teste`. Guarde o resultado em `matriz_teste`.

In [ ]:
# ✏️ Exercício 3
# SEU CÓDIGO AQUI
matriz_teste = ...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
assert matriz_teste.shape == (len(X_teste), matriz_treino.shape[1]), (
    "o teste deve ter o mesmo número de colunas do treino; você usou fit_transform em vez de transform?"
)
print("Exercício 3 correto:", matriz_teste.shape)

**Pergunta escrita.** Suponha que alguém use `pre.fit_transform(X_teste)` no exercício 3. O código roda sem erro. O que estaria errado, e em que situação a matriz resultante teria um número de colunas diferente da matriz de treino?

✏️ **Sua resposta:**

_Escreva aqui._

## 6. `Pipeline`: pré-processamento e modelo em um objeto só

Até aqui teríamos que lembrar de aplicar o pré-processamento na ordem certa, primeiro com `fit_transform` no treino, depois com `transform` no teste, e só então treinar o modelo. É trabalhoso e fácil de errar.

O `Pipeline` encadeia tudo. Quando chamamos `pipeline.fit(X, y)`, ele faz `fit_transform` em cada passo de pré-processamento e `fit` no modelo final. Quando chamamos `pipeline.predict(X)`, ele faz apenas `transform` nos passos intermediários e `predict` no final. **A regra "fit só no treino" passa a ser garantida pela estrutura do código**, e não pela memória de quem programa.

Como vamos criar vários pré-processadores ao longo da aula, embrulhamos a montagem em uma função. Cada chamada devolve um objeto novo, ainda não ajustado.

In [ ]:
def criar_pre(numericas=NUMERICAS, categoricas=CATEGORICAS, extras=()):
    """Devolve um ColumnTransformer novo. `extras` aceita grupos adicionais no formato (nome, transformador, colunas)."""
    tratar_numericas = Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ])
    tratar_categoricas = Pipeline([
        ("imputar", SimpleImputer(strategy="constant", fill_value="desconhecido")),
        ("codificar", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50)),
    ])
    return ColumnTransformer([
        ("num", tratar_numericas, list(numericas)),
        ("cat", tratar_categoricas, list(categoricas)),
        *extras,
    ])

### O modelo: regressão logística

Apesar do nome, a regressão logística é um modelo de **classificação**. Ela calcula uma soma ponderada das colunas de entrada e passa o resultado por uma função em forma de S, que o converte em uma probabilidade entre 0 e 1:

$$P(\text{não recomenda}) = \frac{1}{1 + e^{-(b + w_1 x_1 + w_2 x_2 + \dots)}}$$

Cada peso $w_i$ diz o quanto aquela coluna empurra a probabilidade para cima (peso positivo) ou para baixo (peso negativo). É um modelo simples, rápido, que funciona bem com muitas colunas esparsas e cujos pesos podem ser lidos. Por isso é o primeiro modelo a tentar em quase todo problema de classificação.

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo = Pipeline([
    ("pre", criar_pre()),
    ("clf", LogisticRegression(max_iter=1000)),
])
modelo

### Um conjunto de validação

Queremos saber se o modelo presta, mas o teste está no cofre. A saída é tirar do próprio treino uma parte para conferência, chamada **conjunto de validação**.

```
 todos os dados ─┬─ treino (80%) ─┬─ ajuste (60% do total)     ← o modelo aprende aqui
                 │                └─ validação (20% do total)  ← conferimos aqui, quantas vezes quisermos
                 └─ teste (20%)                                 ← cofre: uma única vez, no fim
```

In [ ]:
X_aj, X_val, y_aj, y_val = train_test_split(
    X_treino, y_treino, test_size=0.25, stratify=y_treino, random_state=SEMENTE
)

modelo.fit(X_aj, y_aj)             # ajusta o pré-processamento E o modelo, só com os dados de ajuste

print("Classes previstas para 8 clientes da validação:", modelo.predict(X_val.head(8)))
print("Probabilidades de [recomenda, não recomenda]:")
print(modelo.predict_proba(X_val.head(4)).round(3))

- `predict` devolve a classe (0 ou 1).
- `predict_proba` devolve uma coluna por classe, na ordem de `modelo.classes_`. A segunda coluna é a probabilidade de "não recomenda".

Por padrão, `predict` responde 1 quando essa probabilidade passa de 0,5. Esse limite, chamado **limiar**, é uma escolha, e voltaremos a ele na seção 12.

Os passos de um pipeline podem ser acessados pelo nome, o que permite inspecionar o que cada um aprendeu.

In [ ]:
mediana_aprendida = modelo.named_steps["pre"].named_transformers_["num"].named_steps["imputar"].statistics_
print("Medianas aprendidas para", NUMERICAS, "→", mediana_aprendida)
print("Número de pesos do modelo:", modelo.named_steps["clf"].coef_.shape[1])

## 7. Medindo: muito além da acurácia

### A referência ingênua

Antes de comemorar qualquer número, precisamos de um ponto de comparação. O `DummyClassifier` com `strategy="most_frequent"` é o "modelo" da pergunta da seção 3: responde sempre a classe mais comum.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

ingenuo = DummyClassifier(strategy="most_frequent").fit(X_aj, y_aj)

print(f"Acurácia do modelo ingênuo:        {accuracy_score(y_val, ingenuo.predict(X_val)):.3f}")
print(f"Acurácia da regressão logística:   {accuracy_score(y_val, modelo.predict(X_val)):.3f}")

As duas acurácias são praticamente iguais. Ou o nosso modelo é inútil, ou a acurácia não está contando a história toda. Para descobrir, abrimos os acertos e erros por tipo.

### A matriz de confusão

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_val, modelo.predict(X_val),
    display_labels=["recomenda", "não recomenda"], cmap="Blues", colorbar=False,
)
plt.title("Regressão logística (limiar 0,5)")
plt.show()

Cada linha é a realidade e cada coluna é a previsão. Com a classe positiva sendo "não recomenda":

| | previsto: recomenda | previsto: não recomenda |
|---|---|---|
| **real: recomenda** | verdadeiro negativo (VN) | falso positivo (FP): alarme falso |
| **real: não recomenda** | falso negativo (FN): insatisfeito que passou | verdadeiro positivo (VP): insatisfeito detectado |

O modelo quase nunca prevê "não recomenda". Ele acerta 73% das vezes pelo mesmo motivo que o modelo ingênuo: dizendo "recomenda" para quase todo mundo. Para o atendimento, que quer encontrar os insatisfeitos, ele é praticamente inútil **com este limiar**.

### Precisão, revocação e F1

Duas perguntas diferentes, duas métricas:

- **Precisão** (*precision*): dos que o modelo apontou como insatisfeitos, quantos realmente eram? $\;\text{VP} / (\text{VP} + \text{FP})$
- **Revocação** (*recall*): dos que realmente eram insatisfeitos, quantos o modelo encontrou? $\;\text{VP} / (\text{VP} + \text{FN})$

Há uma tensão entre as duas. Apontar todo mundo como insatisfeito dá revocação de 100% e precisão de 27%. Apontar só os casos óbvios dá precisão alta e revocação baixa. O **F1** é a média harmônica das duas e só é alto quando ambas são altas.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_val, modelo.predict(X_val),
    target_names=["recomenda", "não recomenda"], digits=3, zero_division=0,
))

A linha "não recomenda" mostra revocação perto de zero. A linha `accuracy` esconde isso completamente.

### Métricas que olham para a probabilidade

Precisão, revocação e F1 dependem do limiar. Outras métricas avaliam diretamente as probabilidades, considerando todos os limiares de uma vez:

- **ROC AUC**: a chance de o modelo dar probabilidade maior a um insatisfeito sorteado do que a um satisfeito sorteado. Vale 0,5 para um modelo que chuta e 1,0 para um modelo perfeito.
- **Precisão média** (*average precision*, AP): resume a curva de precisão por revocação. Para um modelo que chuta, vale a proporção de positivos (aqui, 0,27). É mais exigente que a ROC AUC quando a classe positiva é minoria.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

proba_val = modelo.predict_proba(X_val)[:, 1]          # probabilidade de "não recomenda"

print(f"ROC AUC:         {roc_auc_score(y_val, proba_val):.3f}   (chute = 0,500)")
print(f"Precisão média:  {average_precision_score(y_val, proba_val):.3f}   (chute = {y_val.mean():.3f})")

Agora a história fica clara: o modelo tabular **sabe alguma coisa** (0,61 é melhor que 0,50), mas pouco. As probabilidades que ele produz raramente passam de 0,5, e por isso `predict` quase nunca responde 1.

Veja o que acontece ao baixar o limiar.

In [ ]:
from sklearn.metrics import precision_score, recall_score

for limiar in (0.5, 0.4, 0.3, 0.2):
    previsto = (proba_val >= limiar).astype(int)
    print(f"limiar {limiar:.1f} → apontados: {previsto.mean():6.1%} · "
          f"precisão: {precision_score(y_val, previsto, zero_division=0):.3f} · "
          f"revocação: {recall_score(y_val, previsto):.3f}")

| Situação | Métrica recomendada |
|---|---|
| Classes equilibradas, erros com custo parecido | acurácia |
| Classe positiva rara, queremos comparar modelos | precisão média (AP) ou ROC AUC |
| Já existe um limiar definido e importa o equilíbrio entre os erros | F1, ou precisão e revocação separadas |
| Os erros têm custos conhecidos | custo total (seção 12) |

### ✏️ Exercício 4 (★): as métricas na mão

A célula abaixo calcula a matriz de confusão com limiar 0,3 e separa os quatro valores. Calcule `precisao`, `revocacao` e `f1` usando apenas `vn`, `fp`, `fn` e `vp`.

Lembrete: $F_1 = 2 \cdot \dfrac{\text{precisão} \cdot \text{revocação}}{\text{precisão} + \text{revocação}}$

In [ ]:
# ✏️ Exercício 4
from sklearn.metrics import confusion_matrix

previsto_03 = (proba_val >= 0.3).astype(int)
vn, fp, fn, vp = confusion_matrix(y_val, previsto_03).ravel()
print(f"VN={vn}  FP={fp}  FN={fn}  VP={vp}")

# SEU CÓDIGO AQUI
precisao = ...
revocacao = ...
f1 = ...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
from sklearn.metrics import f1_score

assert np.isclose(precisao, precision_score(y_val, previsto_03)), "precisão: dos apontados, quantos eram positivos?"
assert np.isclose(revocacao, recall_score(y_val, previsto_03)), "revocação: dos positivos, quantos foram apontados?"
assert np.isclose(f1, f1_score(y_val, previsto_03)), "confira a fórmula da média harmônica"
print(f"Exercício 4 correto: precisão {precisao:.3f} · revocação {revocacao:.3f} · F1 {f1:.3f}")

## 8. Validação cruzada

Avaliar em um único conjunto de validação tem dois defeitos: o resultado depende de quais linhas caíram na validação, e um quarto dos dados de treino fica sem ser usado para aprender.

A **validação cruzada em k dobras** resolve os dois. O treino é dividido em k partes; o modelo é treinado k vezes, cada vez deixando uma parte diferente de fora para avaliação.

```
 dobra 1:  [ VAL ][ ajuste ][ ajuste ][ ajuste ][ ajuste ]  → nota 1
 dobra 2:  [ ajuste ][ VAL ][ ajuste ][ ajuste ][ ajuste ]  → nota 2
 dobra 3:  [ ajuste ][ ajuste ][ VAL ][ ajuste ][ ajuste ]  → nota 3
 dobra 4:  [ ajuste ][ ajuste ][ ajuste ][ VAL ][ ajuste ]  → nota 4
 dobra 5:  [ ajuste ][ ajuste ][ ajuste ][ ajuste ][ VAL ]  → nota 5
                                             média ± desvio das 5 notas
```

Ganhamos uma estimativa mais estável **e** uma medida de quanto ela varia. O teste continua no cofre: a validação cruzada acontece inteiramente dentro do treino.

Como o pipeline inteiro é passado para a função, em cada dobra o pré-processamento é reajustado apenas com as partes de ajuste daquela dobra.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)   # 5 dobras, mantendo a proporção do alvo

resultado = cross_validate(
    modelo, X_treino, y_treino,
    cv=cv, scoring=["roc_auc", "average_precision", "f1"], n_jobs=-1,
)
pd.DataFrame(resultado).round(3)

Cada linha é uma dobra. As cinco notas de ROC AUC são muito parecidas, sinal de que a estimativa é estável. Se variassem de 0,55 a 0,70, uma diferença de 0,02 entre dois modelos não significaria nada.

Para comparar vários modelos ao longo da aula, criamos uma função que roda a validação cruzada e acumula os resultados em um placar.

In [ ]:
METRICAS = ["roc_auc", "average_precision", "f1"]
placar = {}

def avaliar(nome, estimador, X=X_treino, y=y_treino):
    """Roda validação cruzada de 5 dobras, guarda as médias no placar e devolve o placar atualizado."""
    r = cross_validate(estimador, X, y, cv=cv, scoring=METRICAS, n_jobs=-1)
    placar[nome] = {
        "roc_auc": r["test_roc_auc"].mean(),
        "± auc": r["test_roc_auc"].std(),
        "precisão média": r["test_average_precision"].mean(),
        "f1": r["test_f1"].mean(),
        "tempo de treino (s)": r["fit_time"].sum(),
    }
    return pd.DataFrame(placar).T.round(3)

avaliar("ingênuo", DummyClassifier(strategy="most_frequent"))
avaliar("tabular: logística", modelo)

Trocar de modelo é trocar um passo do pipeline. O pré-processamento continua o mesmo.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelo_arvore = Pipeline([
    ("pre", criar_pre()),
    ("clf", DecisionTreeClassifier(max_depth=5, random_state=SEMENTE)),
])
avaliar("tabular: árvore (prof. 5)", modelo_arvore)

### ✏️ Exercício 5 (★★): mais colunas ajudam?

Temos duas colunas categóricas ainda não usadas: `site_category_lv2` (a subcategoria do produto, com centenas de valores) e `dia_semana`.

1. Crie `modelo_ex5`, um pipeline de regressão logística cujo pré-processador inclua essas duas colunas **além** das categóricas atuais. Use a função `criar_pre`.
2. Avalie com `avaliar("tabular: + subcategoria e dia", modelo_ex5)`.

In [ ]:
# ✏️ Exercício 5
# SEU CÓDIGO AQUI
modelo_ex5 = ...
avaliar("tabular: + subcategoria e dia", modelo_ex5)

In [ ]:
# ✅ Verificação do exercício 5 (não altere)
assert "tabular: + subcategoria e dia" in placar, "chame avaliar(...) com exatamente esse nome"
assert placar["tabular: + subcategoria e dia"]["roc_auc"] > placar["tabular: logística"]["roc_auc"] + 0.02, (
    "a AUC deveria subir alguns pontos; as duas colunas novas entraram em `categoricas`?"
)
print("Exercício 5 correto.")
pd.DataFrame(placar).T.round(3)

**Pergunta escrita.** A subcategoria tem centenas de valores distintos. Que opção do `OneHotEncoder`, já configurada em `criar_pre`, evita que isso vire um problema? O que aconteceria sem ela?

✏️ **Sua resposta:**

_Escreva aqui._

## 9. Vazamento de dados

**Vazamento de dados** (*data leakage*) acontece quando o modelo, durante o treino, tem acesso a informação que **não estará disponível quando ele for usado de verdade**. O efeito é sempre o mesmo: a avaliação fica ótima e o modelo decepciona em produção. É o erro mais caro em projetos de aprendizagem de máquina, porque não produz mensagem de erro. Produz um número bonito.

Os três tipos mais comuns:

| Tipo | O que acontece | Exemplo |
|---|---|---|
| **1. Pré-processamento fora da validação** | uma transformação aprende com linhas que depois serão usadas para avaliar | selecionar as melhores colunas usando o conjunto inteiro e só depois validar |
| **2. Alvo disfarçado** | uma coluna contém o alvo, ou algo calculado a partir dele | "taxa de reclamação do produto" calculada incluindo a própria linha |
| **3. Estrutura ignorada** | linhas de treino e de avaliação não são independentes | o mesmo cliente nos dois lados; treinar com o futuro para prever o passado |

Vamos provocar os dois primeiros de propósito, medir o estrago e consertar.

### Demonstração A: seleção de colunas fora da validação

Vamos criar dados **sem nenhum sinal**: 100 linhas, 5.000 colunas de ruído e um alvo sorteado na moeda. Nenhum modelo honesto pode acertar mais de 50% aqui.

O procedimento com vazamento: escolher as 20 colunas mais associadas ao alvo usando **todas as linhas** e depois fazer a validação cruzada só com essas 20.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(0)
X_ruido = rng.normal(size=(100, 5000))        # ruído puro
y_moeda = rng.integers(0, 2, size=100)        # cara ou coroa

# ⚠️ COM VAZAMENTO: a seleção enxerga todas as linhas, inclusive as que serão usadas para validar
seletor = SelectKBest(f_classif, k=20).fit(X_ruido, y_moeda)
X_selecionado = seletor.transform(X_ruido)
acc_vazando = cross_val_score(LogisticRegression(), X_selecionado, y_moeda, cv=5).mean()

print(f"Acurácia com a seleção feita fora da validação: {acc_vazando:.2f}")

### 🔮 Preveja antes de rodar

Os dados são ruído. De onde vem uma acurácia tão alta? E qual será a acurácia se a seleção de colunas for um passo do pipeline?

In [ ]:
from sklearn.pipeline import make_pipeline     # atalho: cria um Pipeline dando nomes automáticos aos passos

# CORRETO: a seleção é refeita dentro de cada dobra, só com as linhas de ajuste
pipeline_ruido = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
acc_correta = cross_val_score(pipeline_ruido, X_ruido, y_moeda, cv=5).mean()

print(f"Acurácia com a seleção dentro do pipeline:      {acc_correta:.2f}")

Entre 5.000 colunas de ruído, sempre haverá 20 que, **por acaso**, acompanham o alvo naquelas 100 linhas. Quando a seleção enxerga todas as linhas, ela escolhe colunas que "funcionam" inclusive nas linhas de validação, e a validação apenas confirma a coincidência. Dentro do pipeline, a coincidência é encontrada nas linhas de ajuste e não se repete nas de validação.

> ⚠️ **Regra prática:** todo passo que tem `fit` e é ajustado com dados que depois avaliam o modelo é um candidato a vazamento. Padronização, imputação, one-hot, seleção de colunas, vocabulário de texto, redução de dimensionalidade: tudo isso vai **dentro** do pipeline.

### Demonstração B: o histórico que contém a resposta

Agora nos dados reais. Alguém do time tem uma ideia razoável: *"produtos ruins recebem mais avaliações negativas; vamos dar ao modelo a taxa de não recomendação de cada produto"*.

In [ ]:
com_historico = df.copy()
com_historico["taxa_produto"] = com_historico.groupby("product_id")[ALVO].transform("mean")   # ⚠️

X_hist = com_historico.drop(columns=[ALVO, "recommend_to_a_friend"]).loc[X_treino.index]

modelo_hist = Pipeline([
    ("pre", criar_pre(numericas=NUMERICAS + ["taxa_produto"])),
    ("clf", LogisticRegression(max_iter=1000)),
])
avaliar("⚠️ histórico do produto (vazando)", modelo_hist, X=X_hist)

A ROC AUC saltou de 0,61 para mais de 0,90 com uma única coluna. Resultados bons demais pedem desconfiança. Vejamos quantas avaliações cada produto tem.

In [ ]:
avaliacoes_por_produto = df["product_id"].value_counts()
print(f"Produtos distintos: {milhar(len(avaliacoes_por_produto))}")
print(f"Produtos com uma única avaliação: {(avaliacoes_por_produto == 1).mean():.0%}")

A maior parte dos produtos tem **uma única avaliação**. Para eles, a "taxa de não recomendação do produto" é 0 ou 1: exatamente a resposta daquela linha. A coluna é o alvo com outro nome.

Repare em dois detalhes:

1. O vazamento aconteceu **mesmo usando pipeline e validação cruzada**, porque a coluna foi calculada antes, com o `groupby` sobre a tabela inteira. O pipeline só protege o que está dentro dele.
2. A ideia original é boa. O histórico do produto é informação legítima, desde que calculado **sem a própria linha e sem linhas de validação**.

O scikit-learn tem um transformador que faz isso corretamente: o `TargetEncoder`. Ele substitui cada categoria pela média do alvo naquela categoria e, durante o `fit_transform`, calcula essa média de forma cruzada, de modo que nenhuma linha participe do cálculo do próprio valor. Como é um passo do pipeline, também respeita as dobras da validação. (O sorteio interno dele não está fixado, então o resultado pode variar na terceira casa decimal entre execuções.)

In [ ]:
from sklearn.preprocessing import TargetEncoder

modelo_te = Pipeline([
    ("pre", criar_pre(extras=[("produto", TargetEncoder(), ["product_id"])])),
    ("clf", LogisticRegression(max_iter=1000)),
])
avaliar("histórico do produto (TargetEncoder)", modelo_te)

O ganho real do histórico do produto existe, mas é de alguns pontos, não de trinta. A diferença entre os dois números é o tamanho da ilusão que teria sido levada para a reunião de resultados.

### Demonstração C: a coluna que depende do cenário

A nota de 1 a 5 (`overall_rating`) está na tabela e ainda não foi usada. Vamos incluí-la.

In [ ]:
modelo_nota = Pipeline([
    ("pre", criar_pre(numericas=NUMERICAS + ["overall_rating"])),
    ("clf", LogisticRegression(max_iter=1000)),
])
avaliar("⚠️ com a nota (depende do cenário)", modelo_nota)

De novo um salto enorme. Isto é vazamento? **Depende de como o modelo será usado.**

- No nosso cenário (seção 2), o modelo vai ler mensagens de e-mail, chat e redes sociais, onde **não existe nota**. Usar a nota no treino é vazamento: o modelo aprenderia a depender de uma coluna que estará vazia em produção.
- Em outro cenário, em que o modelo só é aplicado a avaliações do formulário, a nota estaria disponível e usá-la seria legítimo (ainda que o modelo fosse pouco interessante: nota baixa, não recomenda).

A pergunta que decide é sempre a mesma:

> **No momento em que o modelo fizer a previsão, este valor já existirá e estará disponível?**

Nenhuma ferramenta responde isso por você. É preciso conhecer o processo que gera os dados.

### ✏️ Exercício 6 (★★): conserte o vazamento

A célula abaixo sorteia rótulos **sem relação alguma** com o texto de 600 avaliações. Mesmo assim, o código consegue uma ROC AUC bem acima de 0,5, porque escolhe as 100 palavras mais associadas ao rótulo usando todas as linhas antes de validar.

Reescreva o procedimento sem vazamento:

1. crie `modelo_ex6`, um pipeline com os mesmos três passos (`CountVectorizer(min_df=2)`, `SelectKBest(chi2, k=100)` e `LogisticRegression(max_iter=1000)`);
2. calcule `auc_correta` com `cross_val_score`, 5 dobras e `scoring="roc_auc"`, passando o **texto bruto** (`amostra["texto"]`) e não a matriz já transformada.

In [ ]:
# ✏️ Exercício 6
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2

amostra = X_treino.sample(600, random_state=1)
y_sorteado = np.random.default_rng(SEMENTE).integers(0, 2, size=len(amostra))   # rótulos aleatórios

# ⚠️ COM VAZAMENTO (não altere este trecho; ele serve de comparação)
contagens = CountVectorizer(min_df=2).fit_transform(amostra["texto"])
escolhidas = SelectKBest(chi2, k=100).fit_transform(contagens, y_sorteado)
auc_vazando = cross_val_score(
    LogisticRegression(max_iter=1000), escolhidas, y_sorteado, cv=5, scoring="roc_auc"
).mean()
print(f"ROC AUC com vazamento: {auc_vazando:.2f}")

# SEU CÓDIGO AQUI
modelo_ex6 = ...
auc_correta = ...

In [ ]:
# ✅ Verificação do exercício 6 (não altere)
assert isinstance(modelo_ex6, Pipeline), "modelo_ex6 deve ser um Pipeline (use make_pipeline ou Pipeline)"
assert len(modelo_ex6.steps) == 3, "o pipeline deve ter três passos: contagem, seleção e modelo"
assert 0.38 < auc_correta < 0.62, "com rótulos aleatórios a AUC honesta fica perto de 0,5; a seleção está dentro do pipeline?"
assert auc_vazando - auc_correta > 0.08
print(f"Exercício 6 correto: {auc_vazando:.2f} com vazamento contra {auc_correta:.2f} sem.")

### ✏️ Exercício 7 (★★): vaza ou não vaza?

Para cada situação, diga se há vazamento e, se houver, de qual dos três tipos. Considere o cenário da seção 2.

1. Preencher os ausentes de `idade` com a mediana calculada sobre a tabela inteira, antes de dividir em treino e teste.
2. Criar a coluna `tamanho_texto` com o número de caracteres da avaliação, antes de dividir.
3. Usar como coluna de entrada a "nota média que o cliente deu em todas as suas avaliações", incluindo a avaliação que está sendo prevista.
4. Treinar com avaliações de março a maio e avaliar com as de janeiro e fevereiro.
5. Remover do treino as avaliações com texto vazio.

✏️ **Sua resposta:**

_Escreva aqui._

## 10. Texto dentro do pipeline

As colunas tabulares dizem pouco sobre a satisfação do cliente. O texto diz quase tudo. Para usá-lo, precisamos convertê-lo em números.

### TF-IDF em um minuto

A forma clássica é contar palavras. Cada palavra do vocabulário vira uma coluna, e cada texto vira uma linha com as contagens. O **TF-IDF** refina a contagem: palavras que aparecem em quase todos os textos ("produto", "de", "comprei") recebem peso baixo, e palavras raras e informativas ("quebrado", "devolução") recebem peso alto.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

frases = ["produto chegou quebrado", "produto ótimo chegou rápido", "ótimo"]
mini = TfidfVectorizer().fit(frases)

pd.DataFrame(mini.transform(frases).toarray(), columns=mini.get_feature_names_out(), index=frases).round(2)

"Quebrado" e "rápido", que aparecem em uma frase só, pesam mais que "produto" e "chegou", que aparecem em duas.

O `TfidfVectorizer` é um transformador como os outros: no `fit` aprende o vocabulário e os pesos, no `transform` aplica. Aprender o vocabulário com o conjunto inteiro seria vazamento do tipo 1, então ele também vai para dentro do pipeline.

> ⚠️ **Um detalhe que causa muitos erros.** Os transformadores de texto esperam uma sequência de textos (uma dimensão), e não uma tabela. Por isso, no `ColumnTransformer`, a coluna de texto é indicada como `"texto"` (uma string), e **não** como `["texto"]` (uma lista).

O argumento `min_df=5` descarta palavras que aparecem em menos de 5 avaliações, como erros de digitação e nomes de produtos.

⏳ A célula abaixo treina cinco modelos com dezenas de milhares de colunas. Leva cerca de meio minuto.

In [ ]:
modelo_texto = Pipeline([
    ("pre", criar_pre(extras=[("txt", TfidfVectorizer(min_df=5), "texto")])),
    ("clf", LogisticRegression(max_iter=1000)),
])
avaliar("tabular + texto", modelo_texto)

A ROC AUC passou de 0,6 para mais de 0,95, usando o mesmo modelo simples. Na maior parte dos projetos, **a informação que entra importa mais que a sofisticação do algoritmo**.

O resultado é parecido com o do modelo que usava a nota. A diferença é que o texto estará disponível em qualquer canal, e a nota não.

### O que o modelo aprendeu?

Como o modelo é linear, cada palavra tem um peso. Pesos positivos empurram para "não recomenda" e negativos, para "recomenda". Vamos ajustar o modelo em todo o treino e olhar os extremos.

In [ ]:
modelo_texto.fit(X_treino, y_treino)

nomes = modelo_texto.named_steps["pre"].get_feature_names_out()
pesos = pd.Series(modelo_texto.named_steps["clf"].coef_[0], index=nomes)
pesos_texto = pesos[pesos.index.str.startswith("txt__")].rename(lambda nome: nome.replace("txt__", ""))

fig, eixos = plt.subplots(1, 2, figsize=(10, 4.5))
pesos_texto.nlargest(15).sort_values().plot.barh(ax=eixos[0], color="#b5482f")
eixos[0].set_title("Empurram para NÃO recomenda")
pesos_texto.nsmallest(15).sort_values(ascending=False).plot.barh(ax=eixos[1], color="#2f6fb5")
eixos[1].set_title("Empurram para recomenda")
for eixo in eixos:
    eixo.set_xlabel("peso na regressão logística")
plt.tight_layout()
plt.show()

As palavras fazem sentido, e isso é uma forma de validação tão importante quanto a métrica. Se entre os maiores pesos aparecesse um identificador, uma data ou o nome de uma coluna derivada do alvo, seria um indício de vazamento.

## 11. Busca de hiperparâmetros

**Hiperparâmetros** são as escolhas feitas antes do treino: a força da regularização do modelo (`C`), o `min_df` do vocabulário, o uso de pares de palavras. O modelo não os aprende sozinho; nós testamos alternativas e comparamos por validação cruzada.

### Como endereçar um hiperparâmetro dentro do pipeline

O nome segue o caminho até o passo, com dois sublinhados entre os níveis: `passo__subpasso__parametro`.

In [ ]:
[nome for nome in modelo_texto.get_params() if nome.startswith(("clf__C", "pre__txt__ngram", "pre__txt__min_df", "pre__cat__codificar__min"))]

Vamos testar duas escolhas:

- `clf__C`: quanto menor, mais forte a regularização (pesos menores, modelo mais simples);
- `pre__txt__ngram_range`: `(1, 1)` usa só palavras isoladas; `(1, 2)` acrescenta pares de palavras vizinhas, o que permite ao modelo distinguir "não gostei" de "gostei".

O `GridSearchCV` testa todas as combinações. São 3 × 2 = 6 combinações, cada uma avaliada em 3 dobras: 18 treinos. Para caber no tempo da aula, a busca usa uma amostra de 30 mil linhas do treino.

> ⚠️ Como a busca é feita com validação cruzada **sobre o pipeline inteiro**, cada combinação é avaliada sem vazamento. E como ela usa apenas o treino, o teste continua intocado.

⏳ Cerca de meio minuto.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV

X_busca = X_treino.sample(30_000, random_state=SEMENTE)
y_busca = y_treino.loc[X_busca.index]

grade = {
    "clf__C": [0.1, 1, 10],
    "pre__txt__ngram_range": [(1, 1), (1, 2)],
}
busca = GridSearchCV(clone(modelo_texto), grade, cv=3, scoring="average_precision", n_jobs=-1)
busca.fit(X_busca, y_busca)

print("Melhor combinação:", busca.best_params_)
(pd.DataFrame(busca.cv_results_)
   [["param_clf__C", "param_pre__txt__ngram_range", "mean_test_score", "std_test_score"]]
   .sort_values("mean_test_score", ascending=False)
   .round(4))

Como ler a tabela:

- compare as diferenças entre as médias com o desvio (`std_test_score`). Se duas combinações diferem menos que o desvio, estão empatadas, e vale escolher a mais simples ou mais rápida;
- se a melhor combinação está na borda da grade (o maior ou o menor valor testado), vale ampliar a grade nessa direção.

`clone` cria uma cópia não ajustada do pipeline, para que a busca não altere o `modelo_texto` que já treinamos.

### ✏️ Exercício 8 (★★★): sua própria busca

Faça uma segunda busca, em `busca_ex8`, sobre **outros dois** hiperparâmetros:

- `pre__txt__min_df` com os valores 2, 5 e 20;
- `clf__class_weight` com os valores `None` e `"balanced"` (a segunda opção dá mais peso aos erros na classe minoritária).

Use os mesmos dados (`X_busca`, `y_busca`), 3 dobras e `scoring="average_precision"`. Parta de `clone(modelo_texto)`.

In [ ]:
# ✏️ Exercício 8
# SEU CÓDIGO AQUI
grade_ex8 = ...
busca_ex8 = ...

In [ ]:
# ✅ Verificação do exercício 8 (não altere)
assert hasattr(busca_ex8, "best_params_"), "faltou chamar busca_ex8.fit(X_busca, y_busca)"
assert set(busca_ex8.best_params_) == {"pre__txt__min_df", "clf__class_weight"}, "a grade deve ter exatamente esses dois hiperparâmetros"
assert len(busca_ex8.cv_results_["params"]) == 6, "eram esperadas 3 × 2 = 6 combinações"
assert busca_ex8.best_score_ > 0.85
print("Exercício 8 correto. Melhor combinação:", busca_ex8.best_params_, f"· AP = {busca_ex8.best_score_:.4f}")
(pd.DataFrame(busca_ex8.cv_results_)
   [["param_pre__txt__min_df", "param_clf__class_weight", "mean_test_score", "std_test_score"]]
   .sort_values("mean_test_score", ascending=False)
   .round(4))

**Pergunta escrita.** Olhe a tabela de resultados acima (nela, `None` aparece como `NaN`). O `class_weight="balanced"` melhorou a precisão média? Por que essa opção costuma mudar muito o F1 com limiar 0,5 e pouco as métricas baseadas em probabilidade?

✏️ **Sua resposta:**

_Escreva aqui._

## 12. Avaliação final: limiar, custo e a abertura do cofre

### O modelo final

Montamos o modelo com a melhor combinação encontrada na busca.

In [ ]:
modelo_final = clone(modelo_texto).set_params(**busca.best_params_)
print("Configuração escolhida:", busca.best_params_)

### Escolhendo o limiar com base em custos

O modelo produz probabilidades. Para agir, alguém precisa decidir a partir de que probabilidade uma mensagem vai para a fila prioritária. Essa decisão depende dos custos de cada erro. Suponha que o time de negócio informou:

- cada mensagem encaminhada ao atendimento prioritário custa **R$ 5** (tempo do atendente), seja o cliente insatisfeito ou não;
- cada cliente insatisfeito que **não** é atendido a tempo custa, em média, **R$ 40** (perda esperada de compras futuras).

O limiar precisa ser escolhido **sem usar o teste**. Para isso usamos `cross_val_predict`: ele devolve, para cada linha do treino, a probabilidade prevista por um modelo que **não viu aquela linha** (a previsão de quando ela estava na dobra de validação).

⏳ Cerca de um minuto.

In [ ]:
from sklearn.model_selection import cross_val_predict

proba_treino = cross_val_predict(
    modelo_final, X_treino, y_treino, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

print(f"ROC AUC (validação cruzada):        {roc_auc_score(y_treino, proba_treino):.4f}")
print(f"Precisão média (validação cruzada): {average_precision_score(y_treino, proba_treino):.4f}")

In [ ]:
CUSTO_ALERTA = 5      # R$ por mensagem encaminhada
CUSTO_PERDA = 40      # R$ por insatisfeito não encaminhado

def custo_total(y_real, proba, limiar, custo_alerta=CUSTO_ALERTA, custo_perda=CUSTO_PERDA):
    """Custo, em reais, de usar `limiar` para decidir quem vai para a fila prioritária."""
    y_real = np.asarray(y_real)
    encaminhado = proba >= limiar
    perdidos = ((y_real == 1) & ~encaminhado).sum()          # falsos negativos
    return custo_alerta * encaminhado.sum() + custo_perda * perdidos

limiares = np.linspace(0.02, 0.98, 49)
custos = [custo_total(y_treino, proba_treino, t) for t in limiares]
limiar_escolhido = limiares[int(np.argmin(custos))]

plt.plot(limiares, np.array(custos) / 1000)
plt.axvline(limiar_escolhido, color="#b5482f", linestyle="--", label=f"menor custo: limiar {limiar_escolhido:.2f}")
plt.axvline(0.5, color="gray", linestyle=":", label="limiar padrão 0.50")
plt.xlabel("limiar")
plt.ylabel("custo no treino (mil R$)")
plt.legend()
plt.show()

print(f"Custo com limiar 0.50:             R$ {milhar(custo_total(y_treino, proba_treino, 0.5))}")
print(f"Custo com limiar {limiar_escolhido:.2f}:             R$ {milhar(custo_total(y_treino, proba_treino, limiar_escolhido))}")
print(f"Sem modelo, encaminhando todas:    R$ {milhar(CUSTO_ALERTA * len(y_treino))}")
print(f"Sem modelo, encaminhando nenhuma:  R$ {milhar(CUSTO_PERDA * int(y_treino.sum()))}")

Como perder um insatisfeito custa oito vezes mais que um encaminhamento, o melhor limiar fica bem abaixo de 0,5: vale a pena aceitar mais alarmes falsos para deixar passar menos insatisfeitos.

A curva de precisão por revocação mostra todas as opções de troca disponíveis. Cada ponto da curva é um limiar.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

previsto_escolhido = proba_treino >= limiar_escolhido
fig, eixo = plt.subplots(figsize=(5.5, 5))
PrecisionRecallDisplay.from_predictions(y_treino, proba_treino, name="modelo final", ax=eixo)
plt.scatter(
    recall_score(y_treino, previsto_escolhido), precision_score(y_treino, previsto_escolhido),
    color="#b5482f", zorder=3, label=f"limiar {limiar_escolhido:.2f}",
)
plt.scatter(
    recall_score(y_treino, proba_treino >= 0.5), precision_score(y_treino, proba_treino >= 0.5),
    color="gray", zorder=3, label="limiar 0.50",
)
plt.xlabel("revocação")
plt.ylabel("precisão")
plt.legend(loc="lower left")
plt.show()

### ✏️ Exercício 9 (★★★): os custos mudaram

O time de negócio reviu as contas: o atendimento prioritário ficou mais caro, **R$ 15** por mensagem, e a perda por insatisfeito não atendido foi reestimada em **R$ 25**.

🔮 Antes de calcular: o novo limiar será maior ou menor que o anterior?

Encontre `limiar_ex9`, o valor de `limiares` que minimiza o custo com os novos números. Use `custo_total` (ela aceita os argumentos `custo_alerta` e `custo_perda`) e as probabilidades `proba_treino`.

In [ ]:
# ✏️ Exercício 9
# SEU CÓDIGO AQUI
custos_ex9 = ...
limiar_ex9 = ...

In [ ]:
# ✅ Verificação do exercício 9 (não altere)
_c = lambda t: custo_total(y_treino, proba_treino, t, custo_alerta=15, custo_perda=25)
assert limiar_ex9 in limiares, "escolha um dos valores de `limiares`"
assert all(_c(limiar_ex9) <= _c(t) for t in limiares), "há um limiar com custo menor; os custos novos foram passados à função?"
assert limiar_ex9 > limiar_escolhido
print(f"Exercício 9 correto: com alerta mais caro, o limiar sobe de {limiar_escolhido:.2f} para {limiar_ex9:.2f}.")

### Abrindo o cofre

Todas as decisões estão tomadas: colunas, modelo, hiperparâmetros e limiar. Só agora treinamos o modelo final com **todo o treino** e avaliamos no teste, **uma única vez**.

⏳ Cerca de 20 segundos.

In [ ]:
modelo_final.fit(X_treino, y_treino)
proba_teste = modelo_final.predict_proba(X_teste)[:, 1]
previsto_teste = (proba_teste >= limiar_escolhido).astype(int)

comparacao = pd.DataFrame({
    "validação cruzada (treino)": [
        roc_auc_score(y_treino, proba_treino),
        average_precision_score(y_treino, proba_treino),
        precision_score(y_treino, previsto_escolhido),
        recall_score(y_treino, previsto_escolhido),
    ],
    "teste": [
        roc_auc_score(y_teste, proba_teste),
        average_precision_score(y_teste, proba_teste),
        precision_score(y_teste, previsto_teste),
        recall_score(y_teste, previsto_teste),
    ],
}, index=["ROC AUC", "precisão média", f"precisão (limiar {limiar_escolhido:.2f})", f"revocação (limiar {limiar_escolhido:.2f})"])
comparacao.round(4)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_teste, previsto_teste,
    display_labels=["recomenda", "não recomenda"], cmap="Blues", colorbar=False,
)
plt.title(f"Modelo final no teste (limiar {limiar_escolhido:.2f})")
plt.show()

Os números do teste ficaram muito próximos dos da validação cruzada. É o resultado esperado quando o processo foi honesto. Um teste **muito pior** que a validação seria sinal de vazamento em alguma etapa ou de que a busca de hiperparâmetros se ajustou demais às dobras.

Compare esta matriz com a da seção 7. O modelo agora encontra a grande maioria dos insatisfeitos.

> ⚠️ A partir daqui o teste está "gasto". Se você voltar, mudar alguma coisa e medir de novo no teste, o novo número já não é uma estimativa imparcial. Em projetos reais, isso significa separar um novo conjunto de teste, de preferência com dados mais recentes.

## 13. Salvar e reutilizar o modelo

Como o pipeline contém o pré-processamento, basta salvar um único objeto. Quem for usá-lo entrega uma tabela com as colunas originais.

In [ ]:
import joblib

joblib.dump(modelo_final, "modelo_avaliacoes.joblib")
recarregado = joblib.load("modelo_avaliacoes.joblib")

novas = pd.DataFrame({
    "texto": [
        "Chegou antes do prazo. Produto excelente, funciona muito bem, recomendo!",
        "Péssimo. Veio com defeito, pedi a troca há um mês e ninguém responde.",
        "Razoável. O produto é bom, mas a entrega atrasou duas semanas.",
    ],
    "idade": [34, 52, np.nan],
    "hora": [10, 23, 15],
    "site_category_lv1": ["Celulares e Smartphones", "Eletroportáteis", "Categoria Nova"],
    "reviewer_state": ["PE", "SP", "PE"],
    "reviewer_gender": ["F", "M", np.nan],
})

novas["prob_nao_recomenda"] = recarregado.predict_proba(novas)[:, 1].round(3)
novas["fila_prioritaria"] = novas["prob_nao_recomenda"] >= limiar_escolhido
novas[["texto", "prob_nao_recomenda", "fila_prioritaria"]]

O pipeline lidou sozinho com a idade ausente, com o gênero ausente e com uma categoria que nunca apareceu no treino. Essas situações, que quebrariam um código de pré-processamento escrito à mão, são exatamente as que aparecem em produção.

> ⚠️ **Dois cuidados com arquivos `.joblib`:**
> - eles só são garantidos na **mesma versão** do scikit-learn em que foram salvos. Registre as versões junto com o modelo (a primeira célula deste notebook serve para isso);
> - carregar um arquivo desses executa código. **Nunca abra um `.joblib` ou `.pkl` de origem desconhecida.**

## 14. Para levar

### Erros comuns

| Erro | Como evitar |
|---|---|
| `fit` ou `fit_transform` no teste | todo o pré-processamento dentro de um `Pipeline` |
| Avaliar só com acurácia em problema desbalanceado | comparar com o `DummyClassifier`; usar AP, ROC AUC, precisão e revocação |
| Usar o teste várias vezes para decidir | decidir com validação cruzada; abrir o teste uma vez |
| Criar colunas com `groupby` sobre o alvo antes de dividir | `TargetEncoder` dentro do pipeline |
| Incluir colunas que não existirão na hora de prever | perguntar, para cada coluna: "isso estará disponível no momento da previsão?" |
| Passar `["texto"]` em vez de `"texto"` ao vetorizador | coluna de texto é indicada como string no `ColumnTransformer` |
| Aceitar o limiar 0,5 sem pensar | escolher o limiar pelos custos, com `cross_val_predict` |

### Resumo

- No scikit-learn, **`fit` aprende** e **`transform`/`predict` aplicam**. O que foi aprendido fica em atributos terminados em `_`.
- O **`ColumnTransformer`** dá a cada tipo de coluna o tratamento adequado; o **`Pipeline`** junta pré-processamento e modelo em um objeto que pode ser validado, ajustado e salvo como uma coisa só.
- Em problemas desbalanceados, a **acurácia engana**. Sempre compare com um modelo ingênuo.
- A **validação cruzada** dá uma estimativa mais estável e mostra a variação. O **teste** fica trancado até o fim.
- **Vazamento** é o modelo ter, no treino, informação que não terá em uso. Desconfie de resultados bons demais.
- **Informação nova** (o texto) rendeu muito mais que trocar de algoritmo.
- O **limiar** é uma decisão de negócio e se escolhe a partir de custos.

### O que vem a seguir

No notebook 06 trocamos a regressão logística por modelos de *gradient boosting*, que costumam ser os melhores para dados tabulares, usamos o Optuna para buscar hiperparâmetros de forma mais inteligente e o SHAP para explicar previsões individuais. Tudo o que você aprendeu aqui sobre pipelines, validação e vazamento continua valendo.

### Para casa

A lista de exercícios desta aula está no notebook `05_casa.ipynb`. Ela usa o mesmo conjunto de dados com um alvo diferente e pede que você tome, sozinho, as decisões que tomamos juntos aqui.